# Thuiszorg Route Algoritme — Heerlen

## Structuur
1. Installaties & imports
2. Data laden uit CSV
3. Geocoding: adressen naar GPS coordinaten
4. Matching: medewerker en client constraints
5. Afstandsmatrix berekenen
6. OR-Tools VRP oplossen
7. Resultaat visualiseren op kaart

## 1. Installaties & Imports

In [2]:
# Uncomment om te installeren indien nodig
%pip install ortools geopy folium pandas

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import math
import time
import folium
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from IPython.display import display

print('Imports geslaagd')

Imports geslaagd


## 2. Data laden uit CSV
Pas de paden aan naar jouw mapstructuur.

In [6]:
# Paden aanpassen indien nodig
EMPLOYEES_CSV = '../backend/employees.csv'
CLIENTEN_CSV  = '../backend/clienten.csv'

# Medewerkers laden
df_mw = pd.read_csv(EMPLOYEES_CSV)
print(f'{len(df_mw)} medewerkers geladen')
print(df_mw[['name','address','time_window_start','time_window_end','dogs','cats','smokes']].to_string())

20 medewerkers geladen
            name                                   address time_window_start time_window_end  dogs  cats  smokes
0         Vickie           Kanaalstraat 11 6418 NI Heerlen             07:00           18:00     0    -1   False
1           Anna               Lindelaan 3 6414 CW Heerlen             07:00           18:00    -1     2    True
2         Lauren        Oude Lindestraat 4 6411 YU Heerlen             07:00           18:00    -1     0   False
3    Juan Manuel  Deken Nicolaijestraat 18 6411 HC Heerlen             07:00           18:00    -1     1   False
4        Paulina             Julianaweg 50 6413 XK Heerlen             07:00           18:00    -1     2   False
5          Zlata          Tollensstraat 31 6416 GK Heerlen             07:00           18:00     0     1   False
6          Mason         Raadhuisstraat 20 6411 HW Heerlen             07:00           18:00    -1    -1    True
7      Hans-Theo                 Bongerd 7 6411 CZ Heerlen             07

In [7]:
# Clienten laden
df_cl = pd.read_csv(CLIENTEN_CSV)

# Volledig adres samenvoegen
df_cl['address'] = df_cl['Straat'] + ' ' + df_cl['Postcode'].astype(str) + ' ' + df_cl['Stad']

# Constraints parsen uit de Opmerkingen kolom
df_cl['heeft_hond'] = df_cl['Opmerkingen'].fillna('').str.contains('Hond', case=False)
df_cl['heeft_kat']  = df_cl['Opmerkingen'].fillna('').str.contains('Kat',  case=False)
df_cl['rookt']      = df_cl['Opmerkingen'].fillna('').str.contains('Rookt', case=False)

print(f'{len(df_cl)} clienten geladen')
print(df_cl[['Naam','address','Duur (min)','Tijdvensters','heeft_hond','heeft_kat','rookt']].to_string())

100 clienten geladen
                 Naam                                          address  Duur (min) Tijdvensters  heeft_hond  heeft_kat  rookt
0         Mw. Nijssen                  Bethlehemstraat 24 6418 Heerlen         120  08:00-12:00       False       True  False
1       Dhr. Lucassen                      Trompstraat 30 6412 Heerlen          60  08:00-12:00       False      False  False
2          Dhr. Bours                   Nazarethstraat 40 6418 Heerlen          60  08:00-12:00       False      False   True
3        Dhr. Hanssen                     Bradleystraat 4 6418 Heerlen         180  12:00-18:00        True      False  False
4         Dhr. Strous                   Hambeukerboord 40 6418 Heerlen         120  08:00-12:00       False       True   True
5         Mw. Dormans  Laan van Hövell tot Westerflier 29 6411 Heerlen          90  08:00-12:00       False       True  False
6      Mw. Steinbusch                 Coriovallumstraat 3 6411 Heerlen          90  08:00-12:00  

## 3. Geocoding: adressen naar GPS coordinaten

We gebruiken Nominatim (OpenStreetMap) om elk adres om te zetten naar lat/lon.

Let op: dit duurt even vanwege de rate limit van 1 request per seconde.
Resultaten worden opgeslagen zodat je dit maar eenmalig hoeft te doen.

In [8]:
geolocator = Nominatim(user_agent='thuiszorg_heerlen_planner')

def geocode_adres(adres, retries=3):
    """Zet een adres om naar (lat, lon). Geeft None terug bij mislukking."""
    for poging in range(retries):
        try:
            locatie = geolocator.geocode(adres, timeout=10)
            if locatie:
                return (locatie.latitude, locatie.longitude)
            # Fallback: probeer alleen postcode + stad
            delen = adres.split(' ')
            if len(delen) > 2:
                adres_kort = ' '.join(delen[-2:])
                locatie = geolocator.geocode(adres_kort, timeout=10)
                if locatie:
                    return (locatie.latitude, locatie.longitude)
        except GeocoderTimedOut:
            time.sleep(2)
    return None

# Geocode medewerkers
print('Bezig met geocoden van medewerkers...')
mw_coords = []
for _, rij in df_mw.iterrows():
    coords = geocode_adres(rij['address'])
    mw_coords.append(coords)
    status = f'OK {coords}' if coords else 'NIET GEVONDEN'
    print(f'  {rij["name"]:20s} -> {status}')
    time.sleep(1)

df_mw['coords'] = mw_coords
print(f'\n{sum(1 for c in mw_coords if c)} / {len(mw_coords)} medewerkers geocoded')

Bezig met geocoden van medewerkers...
  Vickie               -> OK (50.8679279, 6.0033227)
  Anna                 -> OK (12.0981319, -68.9151601)
  Lauren               -> OK (50.8830154, 5.9781217)
  Juan Manuel          -> OK (50.8847295, 5.9772287)
  Paulina              -> NIET GEVONDEN
  Zlata                -> OK (50.886033, 6.0006179)
  Mason                -> OK (50.8865441, 5.9771527)
  Hans-Theo            -> OK (50.8882948, 5.9789622)
  Lillian              -> OK (50.8846877, 5.9804978)
  Catalina             -> NIET GEVONDEN
  Isabella             -> OK (50.8866664, 5.9823367)
  Mark-Jan             -> OK (50.8822054, 5.9775497)
  Alma                 -> OK (50.8676151, 6.0055626)
  Isabelle             -> OK (50.8675543, 6.0037273)
  Monica               -> OK (50.8624618, 6.0039026)
  Eino                 -> OK (50.8829326, 5.9784556)
  Alf                  -> OK (50.870093, 6.0003524)
  Alvina               -> OK (50.8753041, 5.9969133)
  employees 19         -> OK (50.8

In [9]:
# Geocode clienten
print('Bezig met geocoden van clienten...')
cl_coords = []
for _, rij in df_cl.iterrows():
    coords = geocode_adres(rij['address'])
    cl_coords.append(coords)
    status = f'OK {coords}' if coords else 'NIET GEVONDEN'
    print(f'  {rij["Naam"]:25s} -> {status}')
    time.sleep(1)

df_cl['coords'] = cl_coords
print(f'\n{sum(1 for c in cl_coords if c)} / {len(cl_coords)} clienten geocoded')

Bezig met geocoden van clienten...
  Mw. Nijssen               -> OK (50.8696937, 6.0085234)
  Dhr. Lucassen             -> OK (50.9298477, 5.9643421)
  Dhr. Bours                -> OK (50.8682437, 6.0080896)
  Dhr. Hanssen              -> OK (50.8712813, 5.9961407)
  Dhr. Strous               -> OK (50.867975, 6.0013467)
  Mw. Dormans               -> OK (50.8834148, 5.980923)
  Mw. Steinbusch            -> OK (50.8856845, 5.9775642)
  Mw. Verstappen            -> OK (50.8876735, 5.9773285)
  Dhr. Keulen               -> OK (50.8822054, 5.9775497)
  Mw. Groenewegen           -> OK (50.8829464, 5.9809894)
  Mw. Gijsen                -> OK (50.8848282, 5.9754409)
  Dhr. Wijnen               -> OK (50.8706053, 6.0045671)
  Mw. Drissen               -> OK (50.8905943, 5.9815235)
  Dhr. Drissen              -> OK (50.8836505, 5.9766499)
  Dhr. Zijlmans             -> OK (50.8708273, 5.9961592)
  Dhr. Creusen              -> OK (50.8907255, 5.9818134)
  Mw. Mertens               -> OK (50.8

In [10]:
# Verwijder rijen zonder coordinaten
df_mw_ok = df_mw[df_mw['coords'].notna()].reset_index(drop=True)
df_cl_ok  = df_cl[df_cl['coords'].notna()].reset_index(drop=True)

print(f'Medewerkers bruikbaar : {len(df_mw_ok)} / {len(df_mw)}')
print(f'Clienten bruikbaar    : {len(df_cl_ok)} / {len(df_cl)}')

Medewerkers bruikbaar : 18 / 20
Clienten bruikbaar    : 100 / 100


## 4. Matching: medewerker en client constraints

Regels vanuit de data:
- `dogs = -1`  -> medewerker allergisch voor honden -> mag NIET naar client met hond
- `cats = -1`  -> medewerker allergisch voor katten -> mag NIET naar client met kat
- `smokes = false` -> medewerker rookt niet -> bij voorkeur niet naar rokende client

Positieve waarden (bijv. dogs = 2) betekenen dat de medewerker prima met honden overweg kan.

In [11]:
def mag_koppelen(mw_rij, cl_rij, strikt_roken=False):
    """
    Geeft True terug als deze medewerker naar deze client kan.
    
    Parameters:
        strikt_roken : als True, gaan niet-rokers nooit naar rokende clienten
    """
    # Hond allergie: dogs = -1 betekent allergisch
    if int(mw_rij['dogs']) == -1 and cl_rij['heeft_hond']:
        return False

    # Kat allergie: cats = -1 betekent allergisch
    if int(mw_rij['cats']) == -1 and cl_rij['heeft_kat']:
        return False

    # Roken (optioneel strikt)
    if strikt_roken and str(mw_rij['smokes']).lower() == 'false' and cl_rij['rookt']:
        return False

    return True


# Bouw matching matrix
match_matrix = []
for _, mw in df_mw_ok.iterrows():
    rij = [mag_koppelen(mw, cl) for _, cl in df_cl_ok.iterrows()]
    match_matrix.append(rij)

# Samenvatting
print('Matching samenvatting per medewerker:')
for mw_idx, mw in df_mw_ok.iterrows():
    toegestaan = sum(match_matrix[mw_idx])
    print(f'  {mw["name"]:15s} -> {toegestaan}/{len(df_cl_ok)} clienten toegestaan')

# Waarschuwingen
print()
for cl_idx, cl in df_cl_ok.iterrows():
    mogelijke_mw = sum(match_matrix[mw_idx][cl_idx] for mw_idx in range(len(df_mw_ok)))
    if mogelijke_mw == 0:
        print(f'WAARSCHUWING: {cl["Naam"]} heeft GEEN geschikte medewerker!')
    elif mogelijke_mw <= 2:
        print(f'Let op: {cl["Naam"]} heeft slechts {mogelijke_mw} geschikte medewerker(s)')

Matching samenvatting per medewerker:
  Vickie          -> 56/100 clienten toegestaan
  Anna            -> 79/100 clienten toegestaan
  Lauren          -> 79/100 clienten toegestaan
  Juan Manuel     -> 79/100 clienten toegestaan
  Zlata           -> 100/100 clienten toegestaan
  Mason           -> 41/100 clienten toegestaan
  Hans-Theo       -> 41/100 clienten toegestaan
  Lillian         -> 100/100 clienten toegestaan
  Isabella        -> 56/100 clienten toegestaan
  Mark-Jan        -> 100/100 clienten toegestaan
  Alma            -> 79/100 clienten toegestaan
  Isabelle        -> 41/100 clienten toegestaan
  Monica          -> 41/100 clienten toegestaan
  Eino            -> 41/100 clienten toegestaan
  Alf             -> 56/100 clienten toegestaan
  Alvina          -> 79/100 clienten toegestaan
  employees 19    -> 100/100 clienten toegestaan
  employees 20    -> 56/100 clienten toegestaan



## 5. Afstandsmatrix berekenen

Haversine afstand (echte aardbol-afstand in meters) tussen alle punten.

Later te vervangen door echte rijafstanden via OSMnx op de wegenkaart van Heerlen.

In [12]:
def haversine(coord1, coord2):
    """Berekent afstand in meters tussen twee GPS coordinaten."""
    R = 6371000
    lat1, lon1 = math.radians(coord1[0]), math.radians(coord1[1])
    lat2, lon2 = math.radians(coord2[0]), math.radians(coord2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def maak_afstandsmatrix(df_mw, df_cl):
    """
    Bouwt NxN matrix in meters.
    Index 0 t/m (n_mw-1) = medewerker depots
    Index n_mw t/m einde = clienten
    """
    alle_coords = list(df_mw['coords']) + list(df_cl['coords'])
    n = len(alle_coords)
    matrix = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = int(haversine(alle_coords[i], alle_coords[j]))
    return matrix, alle_coords


afstand_matrix, alle_coords = maak_afstandsmatrix(df_mw_ok, df_cl_ok)
n_mw = len(df_mw_ok)
n_cl = len(df_cl_ok)

print(f'Afstandsmatrix klaar: {len(alle_coords)}x{len(alle_coords)}')
print(f'  {n_mw} medewerkers + {n_cl} clienten')
print(f'  Voorbeeld: {df_mw_ok.iloc[0]["name"]} -> {df_cl_ok.iloc[0]["Naam"]}: {afstand_matrix[0][n_mw]:.0f} meter')

Afstandsmatrix klaar: 118x118
  18 medewerkers + 100 clienten
  Voorbeeld: Vickie -> Mw. Nijssen: 414 meter


## 6. OR-Tools VRP oplossen

- Elke medewerker = voertuig met eigen depot (woonadres)
- Elke client = stop die bezocht moet worden
- Matching constraints worden afgedwongen via VehicleVar
- Doel: minimaliseer totale reisafstand met gebalanceerde werkdruk

In [13]:
def los_vrp_op(afstand_matrix, match_matrix, n_mw, n_cl, max_tijd=30):
    """
    Lost het Vehicle Routing Problem op met OR-Tools.

    Parameters:
        afstand_matrix : NxN matrix met afstanden in meters
        match_matrix   : n_mw x n_cl boolean matrix (True = toegestaan)
        n_mw           : aantal medewerkers
        n_cl           : aantal clienten
        max_tijd       : maximale rekentijd in seconden

    Returns:
        routes : dict {mw_idx: [lijst van cl_idx]}
        status : string
    """
    n_nodes = n_mw + n_cl
    depots  = list(range(n_mw))

    manager = pywrapcp.RoutingIndexManager(n_nodes, n_mw, depots, depots)
    routing = pywrapcp.RoutingModel(manager)

    # Afstandsfunctie registreren
    def afstand_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node   = manager.IndexToNode(to_index)
        return afstand_matrix[from_node][to_node]

    transit_idx = routing.RegisterTransitCallback(afstand_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    # Afstandsdimensie voor balancering van werkdruk
    routing.AddDimension(transit_idx, 0, 10_000_000, True, 'Afstand')
    routing.GetDimensionOrDie('Afstand').SetGlobalSpanCostCoefficient(100)

    # Matching constraints via VehicleVar
    # Elke client krijgt alleen de medewerkers toegewezen die mogen
    for cl_idx in range(n_cl):
        cl_node = n_mw + cl_idx
        cl_routing_idx = manager.NodeToIndex(cl_node)
        toegestane_mw = [mw_idx for mw_idx in range(n_mw) if match_matrix[mw_idx][cl_idx]]
        if toegestane_mw:
            routing.VehicleVar(cl_routing_idx).SetValues(toegestane_mw)
        # Als geen enkele mw toegestaan is, laat OR-Tools zelf kiezen (fallback)

    # Zoekstrategie
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    params.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    params.time_limit.seconds = max_tijd

    print(f'OR-Tools bezig... (max {max_tijd} seconden)')
    oplossing = routing.SolveWithParameters(params)

    if not oplossing:
        return {}, 'GEEN OPLOSSING'

    # Routes uitlezen
    routes = {}
    for mw_idx in range(n_mw):
        index = routing.Start(mw_idx)
        route = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            if node >= n_mw:
                route.append(node - n_mw)
            index = oplossing.Value(routing.NextVar(index))
        routes[mw_idx] = route

    status_map = {1: 'OPTIMAAL', 2: 'GOED GENOEG'}
    status = status_map.get(routing.status(), f'STATUS CODE {routing.status()}')
    return routes, status


routes, status = los_vrp_op(afstand_matrix, match_matrix, n_mw, n_cl, max_tijd=15)

print(f'\nResultaat: {status}')
print('\n--- Routes per medewerker ---')
for mw_idx, route in routes.items():
    mw_naam = df_mw_ok.iloc[mw_idx]['name']
    stops   = [df_cl_ok.iloc[i]['Naam'] for i in route]
    zorg    = sum(df_cl_ok.iloc[i]['Duur (min)'] for i in route)
    print(f'  {mw_naam:15s} -> {len(route)} stops, {zorg} min: {" -> ".join(stops) if stops else "(geen stops)"}')

OR-Tools bezig... (max 15 seconden)

Resultaat: OPTIMAAL

--- Routes per medewerker ---
  Vickie          -> 4 stops, 450 min: Mw. Vullings -> Dhr. Bours -> Mw. Hendriks -> Mw. Ruijters
  Anna            -> 0 stops, 0 min: (geen stops)
  Lauren          -> 0 stops, 0 min: (geen stops)
  Juan Manuel     -> 1 stops, 150 min: Dhr. Lenssen
  Zlata           -> 1 stops, 180 min: Mw. Dubois
  Mason           -> 0 stops, 0 min: (geen stops)
  Hans-Theo       -> 0 stops, 0 min: (geen stops)
  Lillian         -> 0 stops, 0 min: (geen stops)
  Isabella        -> 2 stops, 210 min: Dhr. Lucassen -> Mw. Timmermans
  Mark-Jan        -> 1 stops, 180 min: Dhr. Keulen
  Alma            -> 37 stops, 4560 min: Mw. Brouns -> Mw. Wolters -> Dhr. Vonken -> Mw. Savelberg -> Mw. Offermans -> Mw. Haesen -> Mw. Nijssen -> Dhr. Donders -> Mw. Dohmen -> Dhr. Wijnen -> Mw. Donders -> Dhr. Quadvlieg -> Dhr. Duchateau -> Mw. Eussen -> Mw. Schrijvers -> Mw. Mertens -> Dhr. Paulssen -> Dhr. Boosten -> Mw. Geurts -> Dh

## 7. Visualisatie op kaart van Heerlen

Interactieve kaart via Folium. Klik op een marker voor details.

In [17]:
KLEUREN = [
    'blue', 'green', 'purple', 'orange', 'red',
    'darkblue', 'darkgreen', 'cadetblue', 'darkpurple',
    'pink', 'lightblue', 'lightgreen', 'gray', 'black',
    'lightgray', 'beige', 'lightred', 'darkred'
]

def visualiseer_op_kaart(df_mw, df_cl, routes):
    """Toont alle routes op een interactieve Folium kaart van Heerlen."""

    kaart = folium.Map(
        location=[50.8884, 5.9799],  # centrum Heerlen
        zoom_start=13,
        tiles='OpenStreetMap'
    )

    for mw_idx, route in routes.items():
        mw    = df_mw.iloc[mw_idx]
        kleur = KLEUREN[mw_idx % len(KLEUREN)]

        # Medewerker depot marker
        folium.Marker(
            location=mw['coords'],
            popup=folium.Popup(
                f"<b>{mw['name']}</b><br>"
                f"{mw['address']}<br>"
                f"Werktijden: {mw['time_window_start']} - {mw['time_window_end']}<br>"
                f"Stops vandaag: {len(route)}",
                max_width=250
            ),
            tooltip=mw['name'],
            icon=folium.Icon(color=kleur, icon='home', prefix='fa')
        ).add_to(kaart)

        if not route:
            continue

        # Route lijn: depot -> clienten -> depot
        route_coords = [mw['coords']]
        for stap, cl_idx in enumerate(route):
            cl = df_cl.iloc[cl_idx]
            route_coords.append(cl['coords'])

            # Client marker
            folium.CircleMarker(
                location=cl['coords'],
                radius=9,
                color=kleur,
                fill=True,
                fill_color=kleur,
                fill_opacity=0.85,
                popup=folium.Popup(
                    f"<b>{cl['Naam']}</b><br>"
                    f"Stop {stap + 1} van {mw['name']}<br>"
                    f"Zorg: {cl['Type Zorg']}<br>"
                    f"Duur: {cl['Duur (min)']} min<br>"
                    f"Tijdvenster: {cl['Tijdvensters']}<br>"
                    f"Opmerkingen: {cl['Opmerkingen'] if pd.notna(cl['Opmerkingen']) else '-'}",
                    max_width=280
                ),
                tooltip=f"{stap + 1}. {cl['Naam']}"
            ).add_to(kaart)

        route_coords.append(mw['coords'])  # terug naar depot

        folium.PolyLine(
            locations=route_coords,
            color=kleur,
            weight=3,
            opacity=0.7,
            tooltip=f"{mw['name']} ({len(route)} stops)"
        ).add_to(kaart)

    return kaart


kaart = visualiseer_op_kaart(df_mw_ok, df_cl_ok, routes)

# Sla op als HTML en open in browser
kaart_pad = '../output/kaart_heerlen.html'
kaart.save(kaart_pad)
print(f'Kaart opgeslagen: {kaart_pad}')

# Optioneel: automatisch openen in browser
import webbrowser, os
webbrowser.open('file://' + os.path.abspath(kaart_pad))


Kaart opgeslagen: ../output/kaart_heerlen.html


True

## 8. Samenvatting

In [15]:
print('=' * 60)
print('PLANNING SAMENVATTING')
print('=' * 60)
print(f'Status           : {status}')
print(f'Medewerkers      : {n_mw}')
print(f'Clienten         : {n_cl}')
print()

totaal_afstand  = 0
totaal_zorg     = 0
ingeplande_cl   = set(cl_idx for route in routes.values() for cl_idx in route)

for mw_idx, route in routes.items():
    mw       = df_mw_ok.iloc[mw_idx]
    stops    = [df_cl_ok.iloc[i]['Naam'] for i in route]
    zorg_min = sum(df_cl_ok.iloc[i]['Duur (min)'] for i in route)
    totaal_zorg += zorg_min

    punten = [mw['coords']] + [df_cl_ok.iloc[i]['coords'] for i in route] + [mw['coords']]
    reis   = sum(haversine(punten[i], punten[i+1]) for i in range(len(punten)-1))
    totaal_afstand += reis

    print(f"{mw['name']}:")
    print(f"  Stops       : {len(route)} clienten")
    print(f"  Zorgtijd    : {zorg_min} min")
    print(f"  Reisafstand : {reis / 1000:.1f} km")
    if stops:
        print(f"  Volgorde    : {' -> '.join(stops)}")
    print()

niet_ingepland = [df_cl_ok.iloc[i]['Naam'] for i in range(n_cl) if i not in ingeplande_cl]

print(f'Totale reisafstand : {totaal_afstand / 1000:.1f} km')
print(f'Totale zorgtijd    : {totaal_zorg} min ({totaal_zorg / 60:.1f} uur)')

if niet_ingepland:
    print(f'\nNiet ingepland ({len(niet_ingepland)}):')
    for naam in niet_ingepland:
        print(f'  - {naam}')
else:
    print('\nAlle clienten zijn ingepland!')

PLANNING SAMENVATTING
Status           : OPTIMAAL
Medewerkers      : 18
Clienten         : 100

Vickie:
  Stops       : 4 clienten
  Zorgtijd    : 450 min
  Reisafstand : 1.0 km
  Volgorde    : Mw. Vullings -> Dhr. Bours -> Mw. Hendriks -> Mw. Ruijters

Anna:
  Stops       : 0 clienten
  Zorgtijd    : 0 min
  Reisafstand : 0.0 km

Lauren:
  Stops       : 0 clienten
  Zorgtijd    : 0 min
  Reisafstand : 0.0 km

Juan Manuel:
  Stops       : 1 clienten
  Zorgtijd    : 150 min
  Reisafstand : 0.0 km
  Volgorde    : Dhr. Lenssen

Zlata:
  Stops       : 1 clienten
  Zorgtijd    : 180 min
  Reisafstand : 0.3 km
  Volgorde    : Mw. Dubois

Mason:
  Stops       : 0 clienten
  Zorgtijd    : 0 min
  Reisafstand : 0.0 km

Hans-Theo:
  Stops       : 0 clienten
  Zorgtijd    : 0 min
  Reisafstand : 0.0 km

Lillian:
  Stops       : 0 clienten
  Zorgtijd    : 0 min
  Reisafstand : 0.0 km

Isabella:
  Stops       : 2 clienten
  Zorgtijd    : 210 min
  Reisafstand : 10.0 km
  Volgorde    : Dhr. Lucassen

---
## Roadmap

| Stap | Wat | Status |
|------|-----|--------|
| 1 | CSV data laden + matching constraints | Klaar |
| 2 | Geocoding: adressen naar GPS | Klaar |
| 3 | OR-Tools VRP met matching | Klaar |
| 4 | Visualisatie op Folium kaart | Klaar |
| 5 | OSMnx: echte rijafstanden Heerlen | Volgende stap |
| 6 | Tijdvenster constraints in VRP | Daarna |
| 7 | Flask API koppeling met dashboard | Daarna |
| 8 | Leaflet kaart in dashboard | Daarna |